# Analysis: A Reference Gallery of Visualisations

This notebook is a **gallery of pre-built visualisations** for the Instacart
case study. It is intended as a _companion_ to `eda.ipynb`; once you have
finished the exploratory walk-through, use this notebook to:

1. See what a "polished" answer to many of the case-study questions can look like.
2. Reuse the chart patterns (ranked bar charts, distributions drawn from
   counts, line charts) on your own slices of the data.
3. Spot interesting relationships you may want to investigate more deeply
   in `customer_behavior_analysis.ipynb`.

### How this notebook is organised

Every chart follows the same three-step recipe:

| Step                  | What happens                                                                                             |
| --------------------- | -------------------------------------------------------------------------------------------------------- |
| **1. Set-up**         | A short markdown cell explains _what_ we are about to compute and _why_ it matters for the business.     |
| **2. Code**           | A small, focused code cell builds the underlying summary table and renders the plot with Plotly Express. |
| **3. Interpretation** | A markdown cell calls out the key takeaway(s) from the chart.                                            |

:::{tip}

All visualizations use Plotly Express with the `"simple_white"` template
for a clean, publication-ready look. Hover over any element in the chart
to see exact values.

:::

These are provided as samples of how to approach the case-study questions.
You are encouraged to explore the data and find your own insights beyond
these examples.


## Set up the environment

We rely on **pandas** for data manipulation and **Plotly Express** for visualisation. The "simple_white" template gives a clean, minimal look that prints well in reports and notebooks.


In [1]:
import pandas as pd
import plotly.express as px

# Set default plotly template
px.defaults.template = "simple_white"

## Load the Data


The Instacart dataset is split across **five CSV files** that need to be joined
together before we can analyse purchase behaviour at the product, aisle, or
department level.

We load each file directly from the public mirror on GitHub. The `orders`
file is distributed as a ZIP archive, so we tell `pandas` to decompress it
on the fly with `compression="zip"`.


In [2]:
# Load all five tables
base_url = "https://raw.githubusercontent.com/subwaymatch/instacart-dataset-2017/refs/heads/main/"

orders = pd.read_csv(base_url + "orders.csv.zip", compression="zip")
order_products = pd.read_csv(base_url + "order_products_train.csv")
products = pd.read_csv(base_url + "products.csv")
aisles = pd.read_csv(base_url + "aisles.csv")
departments = pd.read_csv(base_url + "departments.csv")

print(f"orders:          {orders.shape}")
print(f"order_products:  {order_products.shape}")
print(f"products:        {products.shape}")
print(f"aisles:          {aisles.shape}")
print(f"departments:     {departments.shape}")

orders:          (3421083, 7)
order_products:  (1384617, 4)
products:        (49688, 4)
aisles:          (134, 2)
departments:     (21, 2)


### Build an enriched order–product table

`order_products` only contains numeric IDs. To analyse behaviour by product
name, aisle, or department, we **merge** it with the lookup tables. The result,
`op`, is the workhorse table for almost every chart in this notebook:

- one row per _item bought_,
- with the product, aisle, and department names attached.

The merge is a chain of three many-to-one joins:

```
order_products  ──►  + product_name, aisle_id, department_id  (via products)
                ──►  + aisle                                   (via aisles)
                ──►  + department                              (via departments)
```


In [3]:
# Merge order_products with product, aisle, and department names
op = (
    order_products.merge(products, on="product_id")
    .merge(aisles, on="aisle_id")
    .merge(departments, on="department_id")
)
op.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,1,49302,1,1,Bulgarian Yogurt,120,16,yogurt,dairy eggs
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs
2,1,10246,3,0,Organic Celery Hearts,83,4,fresh vegetables,produce
3,1,49683,4,0,Cucumber Kirby,83,4,fresh vegetables,produce
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods


## Orders by Day of Week


Knowing **when** customers shop is the first lens through which to view any
retail dataset. We aggregate `orders` by `order_dow` (day of week, stored as
a number from 0 to 6) and plot the result as a bar chart.

:::{caution} Day names are an assumption

Instacart's data dictionary does not say which number is which day. The
`dow_map` dictionary below uses a common community assumption,
**0 = Saturday, 1 = Sunday, ..., 6 = Friday**, so the axis is readable.
Treat the day names as a label of convenience, not a fact.

:::

In [4]:
# Assumed mapping: Instacart did not document which code is which day
dow_map = {
    0: "Saturday",
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
}
dow_counts = (
    orders["order_dow"]
    .map(dow_map)
    .value_counts()
    .reindex([dow_map[i] for i in range(7)])
    .reset_index()
)
dow_counts.columns = ["Day of Week", "Number of Orders"]

fig = px.bar(
    dow_counts,
    x="Day of Week",
    y="Number of Orders",
    title="Number of Orders by Day of Week",
    text_auto=True,
)
fig.show()

> **Takeaway:** Codes 0 and 1 have about 600,000 and 590,000 orders, roughly
> 25% to 40% more than each of the other five days. Under the assumed mapping
> these are Saturday and Sunday, so ordering leans toward the _weekend_. That is
> useful for staffing, promotion timing, and notification scheduling, but
> confirm the day mapping with the data owner before acting on specific days.

## Orders by Hour of Day


Within a day, when do orders cluster? We pivot from day-of-week to
**hour-of-day** by counting `order_hour_of_day` directly. The resulting
distribution is the natural input for any staffing or fulfilment-capacity model.


In [5]:
hour_counts = orders["order_hour_of_day"].value_counts().sort_index().reset_index()
hour_counts.columns = ["Hour of Day", "Number of Orders"]

fig = px.bar(
    hour_counts,
    x="Hour of Day",
    y="Number of Orders",
    title="Number of Orders by Hour of Day",
    text_auto=True,
)
fig.show()

> **Takeaway:** Order volume ramps up sharply from 7 AM, stays near its peak
> from **10 AM to 4 PM** (10 AM is the busiest hour by a small margin), then
> tapers off in the evening. The early morning and late night are very quiet,
> so consider this when scheduling fulfilment shifts or push-notification
> campaigns.

## Top 20 Most Ordered Products


Which individual products carry the most volume? `value_counts()` on
`product_name` gives us the unit-sales ranking; `head(20)` keeps the chart
readable. We rotate the bars horizontally so long product names stay legible.


In [6]:
top_products = op["product_name"].value_counts().head(20).reset_index()
top_products.columns = ["Product", "Number of Orders"]

fig = px.bar(
    top_products,
    x="Number of Orders",
    y="Product",
    orientation="h",
    title="Top 20 Most Ordered Products",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** The leaderboard is dominated by **fresh produce**: 19 of the
> top 20 products are fruits or vegetables (bananas, strawberries, spinach,
> avocados, lemons), and the only exception is organic whole milk. Many are
> organic. These everyday staples are exactly the items a "buy again"
> recommender should surface first.

## Top 15 Departments by Number of Ordered Items


Stepping up one level of the product hierarchy, we count items per
**department**. This tells us which broad sections of the catalogue drive
the most volume, which is handy for assortment, layout, and inventory decisions.


In [7]:
dept_counts = op["department"].value_counts().head(15).reset_index()
dept_counts.columns = ["Department", "Number of Items Ordered"]

fig = px.bar(
    dept_counts,
    x="Number of Items Ordered",
    y="Department",
    orientation="h",
    title="Top 15 Departments by Number of Ordered Items",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** _Produce_ alone accounts for about 409,000 items, nearly twice
> the next department, _dairy eggs_ (about 217,000). Together they make up
> about 45% of all items ordered. If you only had room to optimise two
> departments, these would be the obvious candidates.

## Top 20 Aisles by Number of Ordered Items


Aisles sit _between_ products and departments in the hierarchy and often give
a more actionable view than either: they group similar items but stay
specific enough to inform shelf and search-result design.


In [8]:
aisle_counts = op["aisle"].value_counts().head(20).reset_index()
aisle_counts.columns = ["Aisle", "Number of Items Ordered"]

fig = px.bar(
    aisle_counts,
    x="Number of Items Ordered",
    y="Aisle",
    orientation="h",
    title="Top 20 Aisles by Number of Ordered Items",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** "Fresh vegetables" and "fresh fruits" are the two volume
> leaders by a wide margin, with about 150,000 items each. The next tier
> (packaged vegetables fruits, yogurt, packaged cheese, sparkling water, milk)
> is also dominated by perishables and drinks, reinforcing the picture of
> Instacart as a _fresh-food_ delivery channel.

## Reorder Ratio by Department


Volume tells us _what people buy_; the **reorder ratio** tells us _what
they keep buying_. For each department we average the binary `reordered`
flag, the fraction of items in that department that the customer had
purchased before.

A high reorder ratio is a proxy for **loyalty**: customers come back to the
same items, suggesting habit-driven (rather than discovery-driven) shopping.


In [9]:
reorder_dept = (
    op.groupby("department")["reordered"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
reorder_dept.columns = ["Department", "Reorder Ratio"]

fig = px.bar(
    reorder_dept,
    x="Reorder Ratio",
    y="Department",
    orientation="h",
    title="Reorder Ratio by Department",
    text_auto=".2f",
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** Routine essentials (_dairy eggs_, _produce_, _beverages_,
> _bakery_) show the highest reorder ratios, about 0.63 to 0.68. Departments
> whose items are bought less often or vary more from trip to trip sit at the
> bottom: _personal care_ is the lowest (about 0.34), just below _pantry_ and
> _international_. The pattern matches Instacart's own observation that the
> most popular aisles are also the most reordered.

## Distribution of Order Sizes (Items per Order)


How many items does a typical Instacart basket contain? We group
`order_products` by `order_id`, count the rows in each group, and plot
the resulting basket-size distribution.


In [10]:
order_sizes = order_products.groupby("order_id").size()

# Count how many orders have each basket size, then plot the counts.
# px.histogram on the 131,209 raw sizes would embed every value in the figure.
size_counts = order_sizes.value_counts().sort_index().reset_index()
size_counts.columns = ["Number of Items in Order", "Number of Orders"]

fig = px.bar(
    size_counts,
    x="Number of Items in Order",
    y="Number of Orders",
    title="Distribution of Items per Order",
)
fig.show()

print(order_sizes.describe().round(1))

count    131209.0
mean         10.6
std           7.9
min           1.0
25%           5.0
50%           9.0
75%          14.0
max          80.0
dtype: float64


> **Takeaway:** The middle half of baskets contains **5 to 14 items** and the
> median basket has 9. A long right tail of large orders (up to 80 items)
> pulls the mean up to about 10.6. Because the distribution is right-skewed,
> report the **median**, not the mean, when you summarise basket size.

## Distribution of Days Since Prior Order


The `days_since_prior_order` column records the gap between a customer's
current order and their previous one. Plotting its distribution reveals the
**cadence** of shopping behaviour. We drop the `NaN` values that correspond
to each user's very first order.


In [11]:
# Count orders for each gap of 0 to 30 days; first orders (NaN) are dropped.
# Plotting the 31 counts keeps the figure small (the raw column has 3.2 million values).
days_counts = (
    orders["days_since_prior_order"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
days_counts.columns = ["Days Since Prior Order", "Number of Orders"]

fig = px.bar(
    days_counts,
    x="Days Since Prior Order",
    y="Number of Orders",
    title="Distribution of Days Since Prior Order",
)
fig.show()

> **Takeaway:** Two clear spikes:
>
> - **7 days**: a sizeable group orders weekly, likely as part of a regular
>   routine. Smaller bumps at 14 and 21 days echo the same weekly rhythm.
> - **30 days**: the tallest bar. The dataset _caps_ the field at 30, so this
>   bar actually represents _"30 or more days"_. It is therefore a mix of true
>   monthly shoppers and lapsed customers, a useful caveat to keep in mind
>   when using this column in any model.

## Reorder Ratio by Hour of Day


Does the _type_ of order change over the course of the day? To find out, we
join the `order_products` table with the order timestamp, group by hour, and
compute the average reorder ratio. This separates _exploratory_ shopping
sessions from _routine_ re-stocking sessions.


In [12]:
# Merge order_products with orders to get hour information
op_hour = order_products.merge(orders[["order_id", "order_hour_of_day"]], on="order_id")
reorder_hour = op_hour.groupby("order_hour_of_day")["reordered"].mean().reset_index()
reorder_hour.columns = ["Hour of Day", "Reorder Ratio"]

fig = px.line(
    reorder_hour,
    x="Hour of Day",
    y="Reorder Ratio",
    title="Reorder Ratio by Hour of Day",
    markers=True,
)
fig.show()

> **Takeaway:** The reorder ratio is highest in the **early morning**, about
> 0.65 between 6 and 8 AM, and lowest around midnight to 3 AM (about 0.57 to
> 0.58). Through the main shopping hours it settles near 0.59. Early-morning
> orders lean slightly more on familiar products, but the whole range is only
> about 8 percentage points, so hour of day is a weak signal of how routine
> an order is.

## Top 20 Most Reordered Products


Filtering `op` to only items where `reordered == 1` and counting product
names tells us **which specific products customers come back for**. This is
the single most useful list for designing a "buy again" surface on the
homepage or app.


In [13]:
reorder_products = (
    op[op["reordered"] == 1]["product_name"].value_counts().head(20).reset_index()
)
reorder_products.columns = ["Product", "Times Reordered"]

fig = px.bar(
    reorder_products,
    x="Times Reordered",
    y="Product",
    orientation="h",
    title="Top 20 Most Reordered Products",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** The reorder leaderboard is almost the same list as the
> top-products chart, in a slightly different order, and it is again
> dominated by _fresh produce_. Bananas lead by a wide margin: about 88% of
> the bananas bought (16,557 of 18,726) were reorders. A handful of staple
> items account for a disproportionate share of all repeat purchases.

## Average Add-to-Cart Position by Department


The `add_to_cart_order` column records the **position** at which an item was
added to a basket (1 = first item added). Averaging it by department reveals
the implicit **shopping order** customers follow: which categories they
think of first, and which get added as afterthoughts.


In [14]:
cart_order_dept = (
    op.groupby("department")["add_to_cart_order"].mean().sort_values().reset_index()
)
cart_order_dept.columns = ["Department", "Avg Add-to-Cart Position"]

fig = px.bar(
    cart_order_dept,
    x="Avg Add-to-Cart Position",
    y="Department",
    orientation="h",
    title="Average Add-to-Cart Position by Department",
    text_auto=".1f",
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** _Alcohol_ is added **first** on average (position about 5.6),
> followed by _beverages_ and _dairy eggs_. _Babies_ items are added **last**
> (about 11.1), after _dry goods pasta_ and _international_. _Bulk_ and
> _missing_ sit in the middle of the table. This ordering can guide in-app
> category positioning, with one caveat: a department's average position also
> depends on the size of the baskets it appears in. Alcohol tends to appear in
> smaller baskets and babies items in larger ones (check this by averaging
> basket size by department), so part of this ranking reflects basket size.

## Summary

- Order volume peaks between 10 AM and 4 PM, and codes 0 and 1 of `order_dow` are the two busiest days (Saturday and Sunday under the assumed mapping).
- Fresh produce dominates both the top-products and top-reordered lists. Produce and dairy eggs together make up about 45% of items.
- Dairy eggs, produce, and beverages have the highest reorder ratios; personal care has the lowest.
- The median basket has 9 items, and gaps between orders spike at 7 days and at the 30-day cap.
- Reorder ratio varies only a little by hour of day, peaking in the early morning.
- Alcohol, beverages, and dairy eggs are added to the cart earliest; babies items are added last.

Every item-level chart here uses the train split, which holds one order per customer (their most recent). See the [case overview](case-overview.md) for what this means for customer-level analysis.